# ENLIGHT Account Validation - Find Missing/Mismatch CA/BP

**Goal:** Identify which CA/BP is missing or extra, and determine whether the issue is in the **Counting Program** or **Validation Program**.

In [ ]:
import pandas as pd
from collections import defaultdict

# ============================================================
# SECTION A: Load Program CSV Export
# ============================================================
program_csv_path = "/tmp/program_accounts.csv"  # <-- UPDATE PATH

print("="*70)
print("STEP 1: LOAD PROGRAM CSV")
print("="*70)

df_program = pd.read_csv(program_csv_path, header=None, names=['Rule', 'Value'])
program_rules = df_program.groupby('Rule')['Value'].apply(set).to_dict()

print(f"Loaded {len(df_program)} records")
print(f"\nProgram Rules Found:")
for rule in sorted(program_rules.keys()):
    print(f"  {rule}: {len(program_rules[rule])} values")

In [ ]:
# ============================================================
# SECTION B: Load Validation Data
# ============================================================
print("\n" + "="*70)
print("STEP 2: LOAD VALIDATION DATA")
print("="*70)

# Option 1: Load from validation CSV export
validation_csv_path = "/tmp/validation_accounts.csv"  # <-- UPDATE PATH

# Option 2: Enter counts manually and provide actual values list
# For Rule 1 - enter values from validation query
validation_rule1_vkont = set([
    # <-- PASTE your validation query results here
    # Example: '1234567890', '1234567891', '1234567892'
])

# For now, we'll use a hybrid approach - load from CSV if available
if validation_csv_path and os.path.exists(validation_csv_path):
    df_validation = pd.read_csv(validation_csv_path, header=None, names=['Rule', 'Value'])
    validation_rules = df_validation.groupby('Rule')['Value'].apply(set).to_dict()
    print(f"Loaded validation from CSV: {len(df_validation)} records")
else:
    # Use manual entry
    validation_rules = {
        'Rule1: Vkont': validation_rule1_vkont,
    }
    print("Using manually entered validation data")

In [ ]:
# ============================================================
# SECTION C: Rule-by-Rule Comparison
# ============================================================
print("\n" + "="*70)
print("STEP 3: RULE-BY-RULE COMPARISON")
print("="*70)

def compare_rules(prog_rules, val_rules, rule_name):
    """Compare validation vs program for a specific rule"""
    prog_values = prog_rules.get(rule_name, set())
    val_values = val_rules.get(rule_name, set())
    
    # Missing in program (in validation, not in program)
    missing_in_prog = val_values - prog_values
    
    # Extra in program (in program, not in validation)
    extra_in_prog = prog_values - val_values
    
    return {
        'prog_count': len(prog_values),
        'val_count': len(val_values),
        'missing_in_prog': missing_in_prog,
        'extra_in_prog': extra_in_prog
    }

# Compare all common rules
all_rules = set(list(program_rules.keys()) + list(validation_rules.keys()))

results = {}
print(f"\n{'Rule':<25} {'Validation':>10} {'Program':>10} {'Missing':>10} {'Extra':>10} {'Issue':<20}")
print("-"*90)

for rule in sorted(all_rules):
    r = compare_rules(program_rules, validation_rules, rule)
    results[rule] = r
    
    # Determine issue type
    if r['missing_in_prog'] and r['extra_in_prog']:
        issue = "Both have mismatches"
    elif r['missing_in_prog']:
        issue = "PROBLEM in COUNTING"
    elif r['extra_in_prog']:
        issue = "PROBLEM in VALIDATION"
    else:
        issue = "MATCH"
    
    print(f"{rule:<25} {r['val_count']:>10} {r['prog_count']:>10} {len(r['missing_in_prog']):>10} {len(r['extra_in_prog']):>10} {issue:<20}")

In [ ]:
# ============================================================
# SECTION D: Investigate Specific Rule
# ============================================================
print("\n" + "="*70)
print("STEP 4: INVESTIGATE SPECIFIC RULE")
print("="*70)

target_rule = "Rule1: Vkont"  # <-- CHANGE THIS to rule with issue

if target_rule not in results:
    print(f"Rule '{target_rule}' not found")
else:
    r = results[target_rule]
    
    print(f"\n{'='*70}")
    print(f"DETAILED ANALYSIS: {target_rule}")
    print(f"{'='*70}")
    
    print(f"\nValidation count: {r['val_count']}")
    print(f"Program count:     {r['prog_count']}")
    print(f"Missing in program: {len(r['missing_in_prog'])}")
    print(f"Extra in program:    {len(r['extra_in_prog'])}")
    
    # Show missing CA/BP
    if r['missing_in_prog']:
        print(f"\n{'='*50}")
        print("MISSING IN PROGRAM (Validation has, Program doesn't)")
        print("LIKELY CAUSE: Counting Program logic issue")
        print(f"{'='*50}")
        print("These CA/BP should be extracted by program but are missing:")
        for ca in sorted(r['missing_in_prog'])[:50]:
            print(f"  {ca}")
        if len(r['missing_in_prog']) > 50:
            print(f"  ... and {len(r['missing_in_prog']) - 50} more")
    
    # Show extra CA/BP
    if r['extra_in_prog']:
        print(f"\n{'='*50}")
        print("EXTRA IN PROGRAM (Program has, Validation doesn't)")
        print("LIKELY CAUSE: Validation Program logic issue")
        print(f"{'='*50}")
        print("These CA/BP are in program but should NOT be extracted:")
        for ca in sorted(r['extra_in_prog'])[:50]:
            print(f"  {ca}")
        if len(r['extra_in_prog']) > 50:
            print(f"  ... and {len(r['extra_in_prog']) - 50} more")

In [ ]:
# ============================================================
# SECTION E: Search for Specific CA/BP
# ============================================================
print("\n" + "="*70)
print("STEP 5: SEARCH FOR SPECIFIC CA/BP")
print("="*70)

search_value = "1234567890"  # <-- CHANGE THIS to CA/BP you want to trace

print(f"Searching for: {search_value}")
print("\n" + "-"*50)

# Check in program
print("\nIn PROGRAM CSV:")
found_in_prog = []
for rule, values in program_rules.items():
    if search_value in values:
        found_in_prog.append(rule)
        print(f"  ✅ Found in {rule}")
if not found_in_prog:
    print("  ❌ NOT FOUND in any program rule")

# Check in validation
print("\nIn VALIDATION data:")
found_in_val = []
for rule, values in validation_rules.items():
    if search_value in values:
        found_in_val.append(rule)
        print(f"  ✅ Found in {rule}")
if not found_in_val:
    print("  ❌ NOT FOUND in any validation rule")

# Diagnosis
print("\n" + "="*50)
print("DIAGNOSIS:")
print("="*50)
if found_in_prog and not found_in_val:
    print(f"❌ CA {search_value} is in PROGRAM but NOT in VALIDATION")
    print("→ Issue: VALIDATION program logic")
    print("→ Action: Check validation SQL logic for this CA")
elif found_in_val and not found_in_prog:
    print(f"❌ CA {search_value} is in VALIDATION but NOT in PROGRAM")
    print("→ Issue: COUNTING PROGRAM logic")
    print("→ Action: Check program extraction logic for this CA")
elif found_in_prog and found_in_val:
    print(f"✅ CA {search_value} is in BOTH")
    print("→ No issue for this CA")
else:
    print(f"❓ CA {search_value} not found in either")

In [ ]:
# ============================================================
# SECTION F: Export Issue Report
# ============================================================
print("\n" + "="*70)
print("STEP 6: EXPORT ISSUE REPORT")
print("="*70)

# Find rules with issues
issues = []
for rule, r in results.items():
    if r['missing_in_prog'] or r['extra_in_prog']:
        if r['missing_in_prog']:
            cause = "COUNTING_PROGRAM"
        else:
            cause = "VALIDATION_PROGRAM"
        issues.append({
            'rule': rule,
            'missing_count': len(r['missing_in_prog']),
            'extra_count': len(r['extra_in_prog']),
            'cause': cause,
            'missing_values': list(r['missing_in_prog'])[:10],
            'extra_values': list(r['extra_in_prog'])[:10]
        })

if issues:
    print(f"\nFound {len(issues)} rules with issues:")
    for issue in issues:
        print(f"\n  Rule: {issue['rule']}")
        print(f"  Cause: {issue['cause']}")
        print(f"  Missing: {issue['missing_count']}, Extra: {issue['extra_count']}")
        print(f"  Sample missing: {issue['missing_values'][:3]}")
else:
    print("No issues found - all rules match!")

---

## Summary

| Scenario | Cause | Action |
|----------|-------|--------|
| CA in Validation, NOT in Program | Counting Program issue | Check program SQL/filter logic |
| CA in Program, NOT in Validation | Validation Program issue | Check validation query logic |
| CA in both | Match | No action needed |

**Next steps:**
1. Run the comparison for all rules
2. Identify which rules have issues
3. Check the specific CA/BP values
4. Compare logic between program and validation